In [ ]:
!pip install transformers torch newsapi-python pandas plotly spacy
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 67.0 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
import pandas as pd
from newsapi import NewsApiClient
from transformers import pipeline
from datetime import datetime, timedelta
import spacy
import plotly.graph_objects as go
import plotly.express as px

# ✅ Read API key from Colab Secrets
from google.colab import userdata
NEWS_API_KEY = userdata.get('NEWS_API_KEY')  # must match the name you gave in secrets

# Quick check
if NEWS_API_KEY:
    print("✅ API key loaded successfully!")
else:
    print("❌ Key not found — check the secret name matches exactly")

# Health topics to track
HEALTH_TOPICS = [
    "flu symptoms",
    "mental health",
    "anxiety",
    "diabetes",
    "heart disease",
    "fitness wellness",
    "sleep disorder",
    "obesity"
]

✅ API key loaded successfully!


In [ ]:
# Load RoBERTa for sentiment
print("Loading RoBERTa...")
# ✅ Replace this line in Cell 3
robert = pipeline("text-classification",
                   model="cardiffnlp/twitter-roberta-base-sentiment",
                   return_all_scores=False)

# Load spaCy for keyword extraction
print("Loading spaCy...")
nlp = spacy.load("en_core_web_sm")

print("✅ All models ready!")

Loading RoBERTa...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/747 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

...

RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

Loading spaCy...
✅ All models ready!


In [ ]:
def fetch_health_news(topics, api_key, days_back=7):
    newsapi = NewsApiClient(api_key=api_key)

    from_date = (datetime.now() - timedelta(days=days_back)).strftime('%Y-%m-%d')
    to_date = datetime.now().strftime('%Y-%m-%d')

    all_records = []

    for topic in topics:
        print(f"Fetching news for: {topic}...")
        try:
            articles = newsapi.get_everything(
                q=topic,
                from_param=from_date,
                to=to_date,
                language='en',
                sort_by='publishedAt',
                page_size=15  # 15 per topic × 8 topics = ~120 total
            )

            for article in articles['articles']:
                all_records.append({
                    'date': article['publishedAt'][:10],
                    'headline': article['title'],
                    'source': article['source']['name'],
                    'topic': topic  # tag which topic this belongs to
                })
        except Exception as e:
            print(f"  ⚠️ Error for '{topic}': {e}")

    df = pd.DataFrame(all_records)
    df = df.drop_duplicates(subset='headline')  # remove duplicate headlines
    print(f"\n✅ Total headlines fetched: {len(df)}")
    return df

news_df = fetch_health_news(HEALTH_TOPICS, NEWS_API_KEY, days_back=7)
news_df.head()

Fetching news for: flu symptoms...
Fetching news for: mental health...
Fetching news for: anxiety...
Fetching news for: diabetes...
Fetching news for: heart disease...
Fetching news for: fitness wellness...
Fetching news for: sleep disorder...
Fetching news for: obesity...

✅ Total headlines fetched: 96


,date,headline,source,topic
0,2026-05-31,Johnson & Johnson's Phase 3 prostate cancer st...,PRNewswire,flu symptoms
1,2026-05-31,"Prepare Nigerians for possible Ebola outbreak,...",The Punch,flu symptoms
2,2026-05-30,The silent symptom of Parkinson’s that can app...,The-independent.com,flu symptoms
3,2026-05-29,If 2026 Was Going to Be the Year You Started R...,Reader's Digest,flu symptoms
4,2026-05-29,Best Vitamin D Supplements for Men Over 40 (Im...,Stanleyiroegbu.com,flu symptoms


In [ ]:
def analyze_sentiment(df):
    print("Running sentiment analysis...")
    labels, scores, sentiment_scores = [], [], []

    # This model uses LABEL_0=negative, LABEL_1=neutral, LABEL_2=positive
    label_map = {
        'LABEL_0': -1,  # negative
        'LABEL_1':  0,  # neutral
        'LABEL_2':  1   # positive
    }

    # Human readable name for the label
    label_name_map = {
        'LABEL_0': 'negative',
        'LABEL_1': 'neutral',
        'LABEL_2': 'positive'
    }

    for i, headline in enumerate(df['headline']):
        try:
            result = robert(headline[:512])[0]
            raw_label = result['label']   # e.g. LABEL_0, LABEL_1, LABEL_2
            score = result['score']

            numeric = label_map.get(raw_label, 0) * score
            human_label = label_name_map.get(raw_label, 'neutral')

            labels.append(human_label)
            scores.append(round(score, 3))
            sentiment_scores.append(round(numeric, 3))

        except Exception as e:
            labels.append('neutral')
            scores.append(0.5)
            sentiment_scores.append(0)

        if (i + 1) % 20 == 0:
            print(f"  Processed {i+1}/{len(df)} headlines...")

    df['sentiment_label'] = labels
    df['confidence'] = scores
    df['sentiment_score'] = sentiment_scores

    print("✅ Sentiment analysis complete!")
    return df

news_df = analyze_sentiment(news_df)
news_df[['headline', 'topic', 'sentiment_label', 'sentiment_score']].head(10)

Running sentiment analysis...
  Processed 20/96 headlines...
  Processed 40/96 headlines...
  Processed 60/96 headlines...
  Processed 80/96 headlines...
✅ Sentiment analysis complete!


,headline,topic,sentiment_label,sentiment_score
0,Johnson & Johnson's Phase 3 prostate cancer st...,flu symptoms,positive,0.645
1,"Prepare Nigerians for possible Ebola outbreak,...",flu symptoms,neutral,0.000
2,The silent symptom of Parkinson’s that can app...,flu symptoms,negative,-0.674
3,If 2026 Was Going to Be the Year You Started R...,flu symptoms,negative,-0.824
4,Best Vitamin D Supplements for Men Over 40 (Im...,flu symptoms,positive,0.682
5,Show HN: Simple news aggregator with source bi...,flu symptoms,neutral,0.000
6,LA Canine Outbreak Caused by Low Vaccination R...,flu symptoms,negative,-0.871
7,Kenya blocks Trump's plan to treat Americans e...,flu symptoms,negative,-0.737
8,US tick activity at highest in a decade. Here’...,flu symptoms,neutral,0.000
9,England's stance on 'concerning' measles risk ...,flu symptoms,negative,-0.687


In [ ]:
def extract_keywords(df):
    print("Extracting keywords...")
    keywords_list = []

    for headline in df['headline']:
        doc = nlp(headline)
        # Extract nouns and proper nouns as keywords
        keywords = [token.text.lower() for token in doc
                    if token.pos_ in ['NOUN', 'PROPN']
                    and len(token.text) > 3
                    and not token.is_stop]
        keywords_list.append(', '.join(keywords[:5]))  # top 5 keywords

    df['keywords'] = keywords_list
    print("✅ Keywords extracted!")
    return df

news_df = extract_keywords(news_df)
news_df[['headline', 'keywords']].head(5)

Extracting keywords...
✅ Keywords extracted!


,headline,keywords
0,Johnson & Johnson's Phase 3 prostate cancer st...,"johnson, johnson, phase, prostate, cancer"
1,"Prepare Nigerians for possible Ebola outbreak,...","nigerians, ebola, outbreak, doctors"
2,The silent symptom of Parkinson’s that can app...,"symptom, parkinson, years, signs"
3,If 2026 Was Going to Be the Year You Started R...,"year, backyard, chickens, news"
4,Best Vitamin D Supplements for Men Over 40 (Im...,"vitamin, supplements, immune, hormone, support"


In [ ]:
# --- Table 1: Daily sentiment per topic ---
daily_topic = news_df.groupby(['date', 'topic']).agg(
    avg_sentiment   = ('sentiment_score', 'mean'),
    headline_count  = ('headline', 'count'),
    positive_count  = ('sentiment_label', lambda x: (x == 'positive').sum()),
    negative_count  = ('sentiment_label', lambda x: (x == 'negative').sum()),
    neutral_count   = ('sentiment_label', lambda x: (x == 'neutral').sum())
).reset_index()

# --- Table 2: Overall topic summary ---
topic_summary = news_df.groupby('topic').agg(
    avg_sentiment   = ('sentiment_score', 'mean'),
    total_headlines = ('headline', 'count'),
    positive_pct    = ('sentiment_label', lambda x: round((x == 'positive').sum()/len(x)*100, 1)),
    negative_pct    = ('sentiment_label', lambda x: round((x == 'negative').sum()/len(x)*100, 1))
).reset_index().sort_values('avg_sentiment')

print("✅ Summary tables ready!")
print("\n--- Topic Summary ---")
print(topic_summary)

✅ Summary tables ready!

--- Topic Summary ---
              topic  avg_sentiment  total_headlines  positive_pct  \
3      flu symptoms      -0.324750               12          16.7   
6           obesity      -0.294900               10          20.0   
7    sleep disorder      -0.171000               12          16.7   
5     mental health      -0.111357               14          21.4   
4     heart disease      -0.089500               10           0.0   
1          diabetes      -0.014154               13          23.1   
0           anxiety       0.055545               11          36.4   
2  fitness wellness       0.056929               14          21.4   

   negative_pct  
3          58.3  
6          60.0  
7          41.7  
5          28.6  
4          10.0  
1          23.1  
0          27.3  
2          14.3  


In [ ]:
# Bar chart: which topic has most negative sentiment
fig = px.bar(
    topic_summary,
    x='topic',
    y='avg_sentiment',
    color='avg_sentiment',
    color_continuous_scale='RdYlGn',  # red=negative, green=positive
    title='Average Sentiment by Health Topic (Last 7 Days)',
    labels={'avg_sentiment': 'Sentiment Score', 'topic': 'Health Topic'}
)
fig.update_layout(xaxis_tickangle=-30)
fig.show()

In [ ]:
# All headlines with sentiment
news_df.to_csv('health_headlines.csv', index=False)

# Daily trend per topic
daily_topic.to_csv('daily_topic_sentiment.csv', index=False)

# Topic-level summary
topic_summary.to_csv('topic_summary.csv', index=False)

print("✅ 3 files exported!")
print()
print("📁 health_headlines.csv       ← Every headline + sentiment label")
print("📁 daily_topic_sentiment.csv  ← Daily trend per topic (MAIN FILE)")
print("📁 topic_summary.csv          ← Overall topic ranking")

✅ 3 files exported!

📁 health_headlines.csv       ← Every headline + sentiment label
📁 daily_topic_sentiment.csv  ← Daily trend per topic (MAIN FILE)
📁 topic_summary.csv          ← Overall topic ranking
